In [6]:
import torch
import torch.nn as nn

class Adapter1x1(nn.Module):
    def __init__(self, in_ch=1, out_ch=3):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        # init to replicate single-channel into 3 channels (so output≈[x,x,x])
        with torch.no_grad():
            # either replicate:
            self.conv.weight.copy_(torch.ones_like(self.conv.weight) / 1.0)
            # or identity-like for mean: set each out channel weight to 1.0
            # For better numeric stability, you might prefer 1/1 so adapter output equals input in each channel

    def forward(self, x):
        return self.conv(x)

# Example wrapper: insert adapter in front of model.forward or replace stem
model = get_timmfrv2('edgenext_x_small')
adapter = Adapter1x1(in_ch=1, out_ch=3)

# Option A: wrap forward
class WrappedModel(nn.Module):
    def __init__(self, adapter, model):
        super().__init__()
        self.adapter = adapter
        self.model = model
    def forward(self, x):
        x = self.adapter(x)            # now (B,3,H,W)
        return self.model(x)

wrapped = WrappedModel(adapter, model)

# Freeze backbone, train adapter only
for p in wrapped.model.parameters():
    p.requires_grad = False
for p in wrapped.adapter.parameters():
    p.requires_grad = True

# Optimizer
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, wrapped.parameters()), lr=1e-3)
# If dataset tiny use lr 1e-3 for adapter; for unfreezing backbone later use lower lr (1e-5 - 1e-4)
import torch
import torch.nn as nn

class Adapter1x1(nn.Module):
    def __init__(self, in_ch=1, out_ch=3):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        # init to replicate single-channel into 3 channels (so output≈[x,x,x])
        with torch.no_grad():
            # either replicate:
            self.conv.weight.copy_(torch.ones_like(self.conv.weight) / 1.0)
            # or identity-like for mean: set each out channel weight to 1.0
            # For better numeric stability, you might prefer 1/1 so adapter output equals input in each channel

    def forward(self, x):
        return self.conv(x)

# Example wrapper: insert adapter in front of model.forward or replace stem
model = get_timmfrv2('edgenext_x_small')
adapter = Adapter1x1(in_ch=1, out_ch=3)

# Option A: wrap forward
class WrappedModel(nn.Module):
    def __init__(self, adapter, model):
        super().__init__()
        self.adapter = adapter
        self.model = model
    def forward(self, x):
        x = self.adapter(x)            # now (B,3,H,W)
        return self.model(x)

wrapped = WrappedModel(adapter, model)

# Freeze backbone, train adapter only
for p in wrapped.model.parameters():
    p.requires_grad = False
for p in wrapped.adapter.parameters():
    p.requires_grad = True

# Optimizer
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, wrapped.parameters()), lr=1e-3)
# If dataset tiny use lr 1e-3 for adapter; for unfreezing backbone later use lower lr (1e-5 - 1e-4)


# Above Cell Loads the Model and Adds a 1X1 adapater for adapting to IRIS IMAGE

In [7]:
model

TimmFRWrapperV2(
  (model): EdgeNeXt(
    (stem): Sequential(
      (0): Conv2d(3, 32, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((32,), eps=1e-06, elementwise_affine=True)
    )
    (stages): Sequential(
      (0): EdgeNeXtStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): ConvBlock(
            (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32)
            (norm): LayerNorm((32,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=32, out_features=128, bias=True)
              (act): GELU(approximate='none')
              (drop1): Dropout(p=0.0, inplace=False)
              (norm): Identity()
              (fc2): Linear(in_features=128, out_features=32, bias=True)
              (drop2): Dropout(p=0.0, inplace=False)
            )
            (drop_path): Identity()
          )
          (1): ConvBlock(
            (conv_dw): Conv2d(32, 32, kernel_size

In [9]:
import sys
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader

# Add the parent directory to path for imports
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

# Import evaluation functions
from Evaluation.evaluation import evaluate_model, IRIS_TEST_FOLDER, IRIS_TRAIN_FOLDER
from Evaluation.utils import plot_scores, write_scores, load_scores


In [19]:
# Configuration parameters
config = {
    'model_name': 'edgenext_x_small',  # Model identifier
    'iris_test_folder': 'masked_dataset_augmented/test',
    'iris_train_folder': 'masked_dataset_augmented/train',
    'output_filename': 'scores_iris.txt',  # Will be saved in LOGS/
    'batch_size': 32,
    'use_cosine_similarity': True,  # True for cosine, False for Euclidean
}

# Create LOGS directory if it doesn't exist
os.makedirs('LOGS', exist_ok=True)

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

Configuration:
  model_name: edgenext_x_small
  iris_test_folder: masked_dataset_augmented/test
  iris_train_folder: masked_dataset_augmented/train
  output_filename: scores_iris.txt
  batch_size: 32
  use_cosine_similarity: True


In [ ]:

evaluate_model(
    model=model,  # Your loaded model
    iris_test_folder= config['iris_test_folder'],
    iris_train_folder= config['iris_train_folder'] ,
    output_filename=config['output_filename'],
    model_name=config['model_name'],
    batch_size=config['batch_size'],
    use_cosine_similarity=config['use_cosine_similarity']
)


 Model loaded and moved to cuda
Total parameters: 2,242,620

Loading datasets...
✓ Test samples: 32324
✓ Train samples: 32524

Generating embeddings...
  Processing test set...
  Processing train set...

✓ Test embeddings: 32324
✓ Train embeddings: 32524

Calculating similarities and saving to scores_iris.txt...
Total comparisons: 1,051,305,776


In [ ]:

scores_file = os.path.join('LOGS', config['output_filename'])

if os.path.exists(scores_file):
     plot_scores(scores_file)
